In [0]:
dbutils.widgets.text("CatalogName", "cd_prod")
CatalogName = dbutils.widgets.get("CatalogName")

dbutils.widgets.text('ExternalLocation_raw',"/mntprod_raw")
ExternalLocation_raw = dbutils.widgets.get('ExternalLocation_raw')

dbutils.widgets.text("ExternalLocationName_silver", "/mntprod_silver")
ExternalLocationName_silver = dbutils.widgets.get("ExternalLocationName_silver")
destination_path = ExternalLocation_silver+'/FACT_DifferentShipto_Smartcards'

In [0]:
%run /Configurations/Init_Scripts

In [0]:
spark.conf.set("spark.sql.catalog.current", CatalogName)

In [0]:
import requests 
import json
import traceback
Client_id = dbutils.secrets.get('ABV_AKV_ADB_SCOPE','Moxie-ClientID')
Client_secret = dbutils.secrets.get('ABV_AKV_ADB_SCOPE','Moxie-Secret')
Username = dbutils.secrets.get('ABV_AKV_ADB_SCOPE','Moxie-UserName')
Password = dbutils.secrets.get('ABV_AKV_ADB_SCOPE','Moxie-Password')
InstanceURL = dbutils.secrets.get('ABV_AKV_ADB_SCOPE','Moxie-InstanceURL')
Domain   = dbutils.secrets.get('ABV_AKV_ADB_SCOPE','Moxie-Domain')
AppServiceURL   = dbutils.secrets.get('ABV_AKV_ADB_SCOPE','Azure-AppService')
access_token = generate_moxie_token(Client_id,Client_secret,Username,Password,Domain)['access_token']

In [0]:
# Fetching a CaseNumber based on CaseId
def fetch_moxie_case_status(case_ids,access_token):
    headers = {
        'Authorization': f'Bearer {access_token}',
        'Content-Type': 'application/json'
    }
    API_RETRY_LIMIT = 5

    formatted_case_ids = ','.join([f"'{case_id}'" for case_id in case_ids])
    soql_query = f"SELECT Id, Status FROM Case WHERE Id IN ({formatted_case_ids})"
    endpoint = f"https://{InstanceURL}/services/data/v57.0/query/?q={soql_query}"


    retry_count = 0
    while retry_count <= API_RETRY_LIMIT:
        case_statuses = {}
        response = requests.get(endpoint, headers=headers)
        if response.status_code == 200:
            response_data = response.json()
            if response_data['totalSize'] > 0:
                case_statuses = {record['Id']: record['Status'] for record in response_data['records']}
                return case_statuses         
            else:
                print("No records found.")
                return None
        elif 500 <= response.status_code <= 599 or response.status_code == 429:
            print("Waiting for 90 seconds...")
            sleep(10)
            retry_count += 1
        else:
            print(f"Error Fetching Case Status")
            return None

In [0]:
df = spark.sql("select distinct penaltymoxiecaseid from promotion.FACT_DifferentShipto_Smartcards where penaltymoxiecaseid is not null and Country = 'US' ")
penaltymoxiecaseid_list = [row['penaltymoxiecaseid'] for row in df.collect()]
case_statuses = fetch_moxie_case_status(penaltymoxiecaseid_list,access_token)

# Convert dictionary to list of Rows
rows = [Row(Id=k, Status=v) for k, v in case_statuses.items()]

# Define schema
schema = StructType([
    StructField("caseid", StringType(), True),
    StructField("status", StringType(), True)
])

# Create DataFrame
df_case_status = spark.createDataFrame(rows, schema)

#  Perform merge operation to table FACT_DifferentShipto_Smartcards
# destination_path = '/mnt/silver/FACT_DifferentShipto_Smartcards'
Deltatbl_UL_Logs = DeltaTable.forPath(spark, destination_path)

(Deltatbl_UL_Logs.alias("tgt")
    .merge(df_case_status.alias("src"),
           f"tgt.PenaltyMoxieCaseId = src.caseid ")
    .whenMatchedUpdate(set={
        "PenaltyMoxieCaseStatus": "src.status",
        "UpdatedBy": lit("ADB_MoxieCaseStatus"),
        "UpdatedDate": lit(current_timestamp())
    })
    .execute()
)


In [0]:
df = spark.sql("select distinct investigationmoxiecaseid from promotion.FACT_DifferentShipto_Smartcards where investigationmoxiecaseid is not null and Country = 'US'")
investigationmoxiecaseid_list = [row['investigationmoxiecaseid'] for row in df.collect()]
case_statuses = fetch_moxie_case_status(investigationmoxiecaseid_list,access_token)
# Convert dictionary to list of Rows
rows = [Row(Id=k, Status=v) for k, v in case_statuses.items()]

# Define schema
schema = StructType([
    StructField("caseid", StringType(), True),
    StructField("status", StringType(), True)
])

# Create DataFrame
df_case_status = spark.createDataFrame(rows, schema)

#  Perform merge operation to table FACT_DifferentShipto_Smartcards
# destination_path = '/mnt/silver/FACT_DifferentShipto_Smartcards'
Deltatbl_UL_Logs = DeltaTable.forPath(spark, destination_path)

(Deltatbl_UL_Logs.alias("tgt")
    .merge(df_case_status.alias("src"),
           f"tgt.investigationmoxiecaseid = src.caseid ")
    .whenMatchedUpdate(set={
        "InvestigationMoxieCaseStatus": "src.status",
        "UpdatedBy": lit("ADB_MoxieCaseStatus"),
        "UpdatedDate": lit(current_timestamp())
    })
    .execute()
)
